### API Routes

This server exposes the following endpoint:

| Endpoint                | Description                  |
|------------------------|------------------------------|
| `/api/v1/arrivals`     | Returns arrival information    |

Before you can query these routes, you’ll need to start the server locally.  
Run the following commands in your terminal:

```bash
# Create and activate a virtual environment
python3 -m venv my_venv
source my_venv/bin/activate

# Install dependencies
pip install -r requirements.txt

# Run the server
python3 server.py


### API Key

To obtain your own API key, go to the [NTA Developer Portal](https://developer.nationaltransport.ie/) and create an account by clicking the **Sign Up** button.

You can register using your `@factored.ai` email address. Once logged in, navigate to the **Products** tab and select **GTFS Realtime**.  
Choose a name for the product and subscribe to it. After subscribing, you’ll find your two API keys under the **Profile** tab.

---

### Storing your API Key (Best Practice)

It is considered best practice not to hard-code your API key directly in your codebase.  
Instead, store it in your `local_settings.py` file and import it when needed:


```python
# local_settings.py
API_KEY = "<your-api-key>"
```



In [1]:
import settings
import local_settings
import gtfs

In [2]:
import requests
import pandas as pd
import json
import os

In [3]:
# Using the correct live URL from settings and API key from local_settings

gtfs_instance = gtfs.GTFS(
    live_url=settings.GTFS_LIVE_URL,  # Using the correct URL from settings
    api_key=local_settings.API_KEY,  # Using API key from local_settings
    rebuild_cache=False  # Set to True if you want to rebuild the cache
)

In [10]:
gtfs_instance._read_agencies()

In [70]:
def get_files_from_path(directory: str, extension: str = '.txt') -> list:
    filenames = []
    for entry in os.listdir(directory):
        full_path = os.path.join(directory, entry)
        _, _extension = os.path.splitext(full_path)
        if os.path.isfile(full_path) and _extension == extension:
            filenames.append(full_path)
    return filenames

def generate_df_from_txt(filename: str):
    try:
        df = pd.read_csv(filename)
        return df  # This will be empty if file is empty, which is fine
    except Exception as e:
        print(f"Error reading {filename}: {e}")
        return pd.DataFrame()  # Always return a DataFrame, never None

In [73]:
dataframes = {}
for file in get_files_from_path('data'):
    df_name = file.replace('data/', '').replace('.txt', '_df')
    df = generate_df_from_txt(file)
    dataframes[df_name] = df
    print('\n')
    print(f"{file}: {df.shape[0]} rows, {df.shape[1]} columns")
    print(f"Columns: {list(df.columns)}")

# Display the first few rows of each dataframe
print("\n" + "="*50)
print("SAMPLE DATA FROM EACH DATAFRAME:")
print("="*50)

for name, df in dataframes.items():
    print(f"\n{name.upper()}:")
    print("-" * 30)
    print(df.head(3))
    print(f"Shape: {df.shape}")
    print()



data/agency.txt: 101 rows, 4 columns
Columns: ['agency_id', 'agency_name', 'agency_url', 'agency_timezone']


data/calendar_dates.txt: 2308 rows, 3 columns
Columns: ['service_id', 'date', 'exception_type']


data/stop_times.txt: 5597882 rows, 9 columns
Columns: ['trip_id', 'arrival_time', 'departure_time', 'stop_id', 'stop_sequence', 'stop_headsign', 'pickup_type', 'drop_off_type', 'timepoint']


data/cache_info.txt: 2 rows, 1 columns
Columns: ['{']


data/shapes.txt: 6566454 rows, 5 columns
Columns: ['shape_id', 'shape_pt_lat', 'shape_pt_lon', 'shape_pt_sequence', 'shape_dist_traveled']
Error reading data/timestamp.txt: No columns to parse from file


data/timestamp.txt: 0 rows, 0 columns
Columns: []


data/trips.txt: 177276 rows, 8 columns
Columns: ['route_id', 'service_id', 'trip_id', 'trip_headsign', 'trip_short_name', 'direction_id', 'block_id', 'shape_id']


data/feed_info.txt: 1 rows, 7 columns
Columns: ['feed_publisher_name', 'feed_publisher_url', 'feed_lang', 'feed_start_dat

In [74]:
dataframes['timestamp_df']

""


In [7]:
BASE_URL = "http://localhost:7341"

# Assuming your API key is stored in a local_settings.py file in the same directory
api_key = local_settings.API_KEY
header = {"X-API-KEY": api_key, "Accept": "application/json"}

try:
    # TODO:
    # 1. Load the available stop numbers from static GTFS data (e.g., stops.txt or via a GTFS utility function).
    #   1.1 Put the static data in a folder called `static`
    # 2. Decide which stops to query—either all, a sample, or by filtering on criteria (e.g., first N stops, or a specific route).
    # 3. Construct a stop_list (list of stop numbers as strings).
    # 4. Use the stop_list to call the /api/v1/arrivals endpoint.
    # 
    # r = requests.get(f"{BASE_URL}/api/v1/arrivals", params={"stop": stop_list}, headers=header)
    stop_id = "1508"
    r = requests.get(f"{BASE_URL}/api/v1/arrivals?stop={stop_id}", headers=header)
    r.raise_for_status()
    data = r.json()
    print("Data was retrieved successfully")
except requests.exceptions.HTTPError as e:
    print(f'HTTP error occurred: {e}')
except requests.exceptions.ConnectionError as e:
    print(f'Connection error occurred: {e}')
except requests.exceptions.RequestException as e:
    print(f'An unexpected error occurred: {e}')

Data was retrieved successfully


In [9]:
print(f"Request data for stop {stop_id}")
data

Request data for stop 1508


{'1508': {'arrivals': [{'agency': 'Bus Átha Cliath – Dublin Bus',
    'headsign': 'Charlestown',
    'real_time_arrival': None,
    'route': 'F3',
    'scheduled_arrival': '2025-10-28T10:53:16'},
   {'agency': 'Bus Átha Cliath – Dublin Bus',
    'headsign': 'IKEA Ballymun',
    'real_time_arrival': None,
    'route': 'F1',
    'scheduled_arrival': '2025-10-28T11:03:09'},
   {'agency': 'Bus Átha Cliath – Dublin Bus',
    'headsign': 'Charlestown',
    'real_time_arrival': None,
    'route': 'F2',
    'scheduled_arrival': '2025-10-28T11:04:47'},
   {'agency': 'Bus Átha Cliath – Dublin Bus',
    'headsign': 'Charlestown',
    'real_time_arrival': None,
    'route': 'F3',
    'scheduled_arrival': '2025-10-28T11:08:16'},
   {'agency': 'Bus Átha Cliath – Dublin Bus',
    'headsign': 'Tyrrelstown',
    'real_time_arrival': None,
    'route': '40D',
    'scheduled_arrival': '2025-10-28T11:14:12'},
   {'agency': 'Bus Átha Cliath – Dublin Bus',
    'headsign': 'IKEA Ballymun',
    'real_time_arr